# 상권 영역 DB 구축 (전처리)

서울시 상권분석서비스 '영역-상권' Shapefile을 읽어, 각 상권의
경계 다각형과 속성을 SQLite DB로 저장한다.

**이 노트북은 한 번만 실행하면 된다.** 결과로 만들어진 `commercial_area.db`를
데이터 소스 클래스(런타임)가 읽어서 point-in-polygon 매칭에 사용한다.

전처리 단계:
1. Shapefile(.shp/.dbf) 직접 파싱 — 외부 라이브러리 불필요
2. 다각형 좌표를 EPSG:5181 → WGS84 위경도로 변환
3. SQLite DB에 상권 속성 + 다각형 좌표(JSON) 저장

왜 이렇게 하나:
- 런타임에 Shapefile 라이브러리(geopandas 등)나 좌표 변환(pyproj)
  의존성이 필요 없도록, 무거운 처리를 전처리에서 끝낸다.
- 런타임 데이터 소스는 이 DB만 읽고 point-in-polygon만 수행한다.

## 0. 설정 — 파일 경로

> 업로드한 Shapefile 5종(.shp .dbf .shx .prj .cpg)이 있는 폴더를 지정한다.
> .shp와 .dbf만 직접 사용하지만, 5개 파일이 같은 폴더에 함께 있어야 한다.

In [1]:
import struct
import csv
import math
import json
import sqlite3
import os

# 경로 설정 (db_builder 폴더에서 실행 기준)
SHP_BASE = "../raw_data/서울시 상권분석서비스(영역-상권)"
DB_PATH = "../DB/commercial_area.db"

SHP_PATH = SHP_BASE + ".shp"
DBF_PATH = SHP_BASE + ".dbf"

print("SHP:", SHP_PATH)
print("DBF:", DBF_PATH)
print("출력 DB:", DB_PATH)
print("SHP 존재:", os.path.exists(SHP_PATH))
print("DBF 존재:", os.path.exists(DBF_PATH))

SHP: ../raw_data/서울시 상권분석서비스(영역-상권).shp
DBF: ../raw_data/서울시 상권분석서비스(영역-상권).dbf
출력 DB: ../DB/commercial_area.db
SHP 존재: True
DBF 존재: True


## 1. 좌표 변환 함수 — EPSG:5181 → WGS84 위경도

상권 Shapefile의 좌표계는 Korea 2000 중부원점(EPSG:5181)으로,
미터 단위 평면좌표(X, Y)다. 우리 시스템의 건물 좌표는 위경도(WGS84)이므로
다각형을 위경도로 변환해서 저장한다.

변환식은 횡축 메르카토르(Transverse Mercator) 투영의 역변환이다.
.prj 파일에 있던 파라미터를 그대로 사용한다:
중앙자오선 127°, 원점위도 38°, False Easting 200000, False Northing 500000,
축척계수 1.0, GRS80 타원체.

In [2]:
def tm5181_to_wgs84(x, y):
    """EPSG:5181 평면좌표(x, y 미터)를 WGS84 위경도(lat, lon)로 변환."""
    a = 6378137.0              # GRS80 장반경
    f = 1 / 298.257222101      # 편평률
    k0 = 1.0                   # 축척계수
    lon0 = math.radians(127.0) # 중앙자오선
    lat0 = math.radians(38.0)  # 원점위도
    FE = 200000.0              # False Easting
    FN = 500000.0              # False Northing

    e2 = f * (2 - f)
    ep2 = e2 / (1 - e2)

    def meridian_arc(lat):
        return a * ((1 - e2/4 - 3*e2**2/64 - 5*e2**3/256) * lat
                  - (3*e2/8 + 3*e2**2/32 + 45*e2**3/1024) * math.sin(2*lat)
                  + (15*e2**2/256 + 45*e2**3/1024) * math.sin(4*lat)
                  - (35*e2**3/3072) * math.sin(6*lat))

    M0 = meridian_arc(lat0)
    M = M0 + (y - FN) / k0

    e1 = (1 - math.sqrt(1 - e2)) / (1 + math.sqrt(1 - e2))
    mu = M / (a * (1 - e2/4 - 3*e2**2/64 - 5*e2**3/256))
    lat1 = (mu + (3*e1/2 - 27*e1**3/32) * math.sin(2*mu)
              + (21*e1**2/16 - 55*e1**4/32) * math.sin(4*mu)
              + (151*e1**3/96) * math.sin(6*mu))

    C1 = ep2 * math.cos(lat1)**2
    T1 = math.tan(lat1)**2
    N1 = a / math.sqrt(1 - e2 * math.sin(lat1)**2)
    R1 = a * (1 - e2) / (1 - e2 * math.sin(lat1)**2)**1.5
    D = (x - FE) / (N1 * k0)

    lat = lat1 - (N1 * math.tan(lat1) / R1) * (D**2/2
            - (5 + 3*T1 + 10*C1 - 4*C1**2 - 9*ep2) * D**4/24
            + (61 + 90*T1 + 298*C1 + 45*T1**2 - 252*ep2 - 3*C1**2) * D**6/720)
    lon = lon0 + (D - (1 + 2*T1 + C1) * D**3/6
            + (5 - 2*C1 + 28*T1 - 3*C1**2 + 8*ep2 + 24*T1**2) * D**5/120) / math.cos(lat1)

    return math.degrees(lat), math.degrees(lon)


# 검증: 서울 종로구 근처 좌표를 넣어 위경도가 서울 범위로 나오는지 확인
test_lat, test_lon = tm5181_to_wgs84(196955.8, 453722.1)
print(f"변환 테스트: 위도 {test_lat:.6f}, 경도 {test_lon:.6f}")
assert 37.4 < test_lat < 37.7, "위도가 서울 범위를 벗어남"
assert 126.7 < test_lon < 127.2, "경도가 서울 범위를 벗어남"
print("좌표 변환 함수 정상")

변환 테스트: 위도 37.583049, 경도 126.965535
좌표 변환 함수 정상


## 2. DBF 파싱 — 상권 속성 읽기

DBF는 dBASE 형식의 단순한 테이블 파일이다. 외부 라이브러리 없이 직접 읽는다.

컬럼:
- `TRDAR_SE_C`  상권 구분 코드
- `TRDAR_SE_1`  상권 구분명 (골목상권/발달상권/전통시장/관광특구)
- `TRDAR_CD`    상권코드 (다른 데이터셋과 연결하는 키)
- `TRDAR_CD_N`  상권명
- `XCNTS_VALU`  상권 중심 X좌표 (EPSG:5181)
- `YDNTS_VALU`  상권 중심 Y좌표
- `SIGNGU_CD`   시군구코드
- `ADSTRD_CD`   행정동코드 (행정동 폴백 시 연결고리)
- `ADSTRD_CD_`  행정동명
- `RELM_AR`     상권 면적(㎡)

In [3]:
def read_dbf(path):
    """DBF 파일을 읽어 레코드(dict) 리스트로 반환한다."""
    with open(path, "rb") as f:
        data = f.read()

    num_records = struct.unpack("<I", data[4:8])[0]
    header_size = struct.unpack("<H", data[8:10])[0]
    record_size = struct.unpack("<H", data[10:12])[0]

    # 필드 정의 (32바이트씩)
    fields = []
    pos = 32
    while data[pos] != 0x0D:
        name = data[pos:pos+11].split(b"\x00")[0].decode("utf-8", errors="replace")
        ftype = chr(data[pos+11])
        flen = data[pos+16]
        fields.append((name, ftype, flen))
        pos += 32

    # 레코드 읽기
    records = []
    for r in range(num_records):
        start = header_size + r * record_size
        rec = data[start:start+record_size]
        offset = 1  # 첫 바이트는 삭제 표시 플래그
        row = {}
        for name, ftype, flen in fields:
            raw = rec[offset:offset+flen].decode("utf-8", errors="replace").strip()
            row[name] = raw
            offset += flen
        records.append(row)
    return records


dbf_records = read_dbf(DBF_PATH)
print("상권 속성 레코드 수:", len(dbf_records))
print()
print("첫 레코드 예시:")
for k, v in dbf_records[0].items():
    print(f"  {k}: {v}")

상권 속성 레코드 수: 1650

첫 레코드 예시:
  TRDAR_SE_C: A
  TRDAR_SE_1: 골목상권
  TRDAR_CD: 3110008
  TRDAR_CD_N: 배화여자대학교(박노수미술관)
  XCNTS_VALU: 197093
  YDNTS_VALU: 453418
  SIGNGU_CD: 11110
  SIGNGU_CD_: 종로구
  ADSTRD_CD: 11110515
  ADSTRD_CD_: 청운효자동
  RELM_AR: 149264


## 3. SHP 파싱 — 상권 경계 다각형 읽기

SHP는 도형(geometry)을 담는 바이너리 파일이다. Polygon 타입이며,
각 상권은 하나 이상의 part(외곽선/구멍)와 점들의 좌표로 구성된다.

SHP의 레코드 순서는 DBF의 레코드 순서와 1:1로 대응한다
(N번째 도형 = N번째 속성).

In [4]:
def read_shp_polygons(path):
    """SHP 파일을 읽어 각 레코드의 다각형을 반환한다.

    반환: 리스트. 각 원소는 part들의 리스트이고, 각 part는
          (x, y) 좌표 튜플의 리스트다 (EPSG:5181 원본 좌표).
          예: polygons[0] = [ [(x,y), (x,y), ...],  # part 0 (외곽선)
                              [(x,y), ...] ]        # part 1 (구멍 등)
    """
    with open(path, "rb") as f:
        data = f.read()

    polygons = []
    pos = 100  # 파일 헤더 100바이트 건너뛰기

    while pos < len(data):
        # 레코드 헤더 (big-endian)
        content_len = struct.unpack(">I", data[pos+4:pos+8])[0] * 2
        shape_type = struct.unpack("<I", data[pos+8:pos+12])[0]

        if shape_type == 5:  # Polygon
            num_parts = struct.unpack("<I", data[pos+44:pos+48])[0]
            num_points = struct.unpack("<I", data[pos+48:pos+52])[0]

            parts_start = pos + 52
            parts = struct.unpack(
                f"<{num_parts}I",
                data[parts_start:parts_start + num_parts*4]
            )

            points_start = parts_start + num_parts*4
            all_points = []
            for p in range(num_points):
                x, y = struct.unpack(
                    "<2d",
                    data[points_start + p*16 : points_start + p*16 + 16]
                )
                all_points.append((x, y))

            # parts 인덱스로 점들을 part별로 나눈다
            rec_parts = []
            for i in range(num_parts):
                s = parts[i]
                e = parts[i+1] if i+1 < num_parts else num_points
                rec_parts.append(all_points[s:e])
            polygons.append(rec_parts)
        else:
            # Polygon이 아닌 레코드 (Null 등) — 빈 다각형으로
            polygons.append([])

        pos += 8 + content_len

    return polygons


shp_polygons = read_shp_polygons(SHP_PATH)
print("상권 다각형 레코드 수:", len(shp_polygons))
print("DBF 레코드 수와 일치:", len(shp_polygons) == len(dbf_records))
print()
# 첫 상권의 다각형 정보
first = shp_polygons[0]
print(f"첫 상권: part {len(first)}개, 첫 part의 점 {len(first[0])}개")
print("첫 part의 처음 3개 점(EPSG:5181):", first[0][:3])

상권 다각형 레코드 수: 1650
DBF 레코드 수와 일치: True

첫 상권: part 1개, 첫 part의 점 199개
첫 part의 처음 3개 점(EPSG:5181): [(196955.8422, 453722.1139), (196958.6692, 453717.022), (196967.1371, 453719.48210000095)]


## 4. 다각형을 위경도로 변환

모든 상권의 모든 점을 EPSG:5181 → WGS84 위경도로 변환한다.
점 수가 많아 시간이 조금 걸릴 수 있다 (상권 1,650개).

In [5]:
polygons_wgs84 = []

for rec_parts in shp_polygons:
    converted_parts = []
    for part in rec_parts:
        converted = []
        for x, y in part:
            lat, lon = tm5181_to_wgs84(x, y)
            # (경도, 위도) 순으로 저장 — GeoJSON 관례와 동일
            converted.append([lon, lat])
        converted_parts.append(converted)
    polygons_wgs84.append(converted_parts)

print("변환 완료:", len(polygons_wgs84), "개 상권")
print()
# 첫 상권 첫 part의 처음 2개 점 (위경도)
print("첫 상권 다각형 처음 2개 점 [경도, 위도]:")
for pt in polygons_wgs84[0][0][:2]:
    print("  ", pt)

변환 완료: 1650 개 상권

첫 상권 다각형 처음 2개 점 [경도, 위도]:
   [126.96553552860632, 37.583049042610355]
   [126.96556755564743, 37.58300317427473]


## 5. SQLite DB로 저장

테이블 `commercial_area` 하나에 상권 속성 + 다각형을 저장한다.
다각형은 좌표 배열이라 JSON 문자열로 직렬화해서 한 컬럼에 넣는다.

런타임 데이터 소스는 이 DB를 읽어, 다각형 JSON을 다시 파싱하고
point-in-polygon 판정에 사용한다.

In [6]:
# 기존 DB가 있으면 지우고 새로 만든다
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# 테이블 생성
cur.execute("""
CREATE TABLE commercial_area (
    trdar_cd       TEXT PRIMARY KEY,   -- 상권코드
    trdar_se_name  TEXT,               -- 상권 구분명 (골목상권 등)
    trdar_name     TEXT,               -- 상권명
    signgu_cd      TEXT,               -- 시군구코드
    adstrd_cd      TEXT,               -- 행정동코드
    adstrd_name    TEXT,               -- 행정동명
    area_sqm       REAL,               -- 상권 면적(㎡)
    center_lon     REAL,               -- 상권 중심 경도
    center_lat     REAL,               -- 상권 중심 위도
    bbox_min_lon   REAL,               -- 경계상자 (빠른 1차 필터용)
    bbox_min_lat   REAL,
    bbox_max_lon   REAL,
    bbox_max_lat   REAL,
    polygon_json   TEXT                -- 다각형 좌표 [[part],[part]...] JSON
)
""")

rows = 0
for rec, parts in zip(dbf_records, polygons_wgs84):
    # 다각형이 비어있으면 건너뛴다
    if not parts or not parts[0]:
        continue

    # 경계상자 계산 (모든 part의 모든 점)
    all_lon = [pt[0] for part in parts for pt in part]
    all_lat = [pt[1] for part in parts for pt in part]
    bbox = (min(all_lon), min(all_lat), max(all_lon), max(all_lat))

    # 상권 중심점도 위경도로 변환
    try:
        cx = float(rec["XCNTS_VALU"])
        cy = float(rec["YDNTS_VALU"])
        center_lat, center_lon = tm5181_to_wgs84(cx, cy)
    except (ValueError, KeyError):
        center_lat, center_lon = None, None

    try:
        area = float(rec["RELM_AR"])
    except (ValueError, KeyError):
        area = None

    cur.execute("""
        INSERT OR REPLACE INTO commercial_area VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (
        rec["TRDAR_CD"],
        rec["TRDAR_SE_1"],
        rec["TRDAR_CD_N"],
        rec["SIGNGU_CD"],
        rec["ADSTRD_CD"],
        rec["ADSTRD_CD_"],
        area,
        center_lon, center_lat,
        bbox[0], bbox[1], bbox[2], bbox[3],
        json.dumps(parts),
    ))
    rows += 1

conn.commit()
print("저장 완료:", rows, "개 상권")

# bbox 컬럼에 인덱스 (1차 필터 속도 향상)
cur.execute("CREATE INDEX idx_bbox ON commercial_area(bbox_min_lon, bbox_max_lon, bbox_min_lat, bbox_max_lat)")
conn.commit()
print("인덱스 생성 완료")

저장 완료: 1650 개 상권
인덱스 생성 완료


## 6. 검증 — 저장된 DB 확인

In [7]:
# 저장된 내용 점검
cur.execute("SELECT COUNT(*) FROM commercial_area")
print("총 상권 수:", cur.fetchone()[0])
print()

# 상권 구분별 개수
cur.execute("SELECT trdar_se_name, COUNT(*) FROM commercial_area GROUP BY trdar_se_name")
print("상권 구분별 개수:")
for name, cnt in cur.fetchall():
    print(f"  {name}: {cnt}")
print()

# 샘플 1건
cur.execute("SELECT trdar_cd, trdar_se_name, trdar_name, adstrd_name, area_sqm FROM commercial_area LIMIT 3")
print("샘플 상권:")
for row in cur.fetchall():
    print("  ", row)
print()

# 다각형 JSON이 제대로 들어갔는지
cur.execute("SELECT trdar_name, polygon_json FROM commercial_area LIMIT 1")
name, pjson = cur.fetchone()
parts = json.loads(pjson)
print(f"'{name}' 다각형: part {len(parts)}개, 첫 part 점 {len(parts[0])}개")
print("  첫 점 [경도,위도]:", parts[0][0])

print()
print("=> commercial_area 테이블 완료. 이어서 인구 데이터를 적재한다.")

총 상권 수: 1650

상권 구분별 개수:
  골목상권: 1090
  관광특구: 6
  발달상권: 249
  전통시장: 305

샘플 상권:
   ('3110008', '골목상권', '배화여자대학교(박노수미술관)', '청운효자동', 149264.0)
   ('3001494', '관광특구', '종로·청계 관광특구', '종로1·2·3·4가동', 653127.0)
   ('3130021', '전통시장', '신설종합시장', '숭인2동', 9800.0)

'배화여자대학교(박노수미술관)' 다각형: part 1개, 첫 part 점 199개
  첫 점 [경도,위도]: [126.96553552860632, 37.583049042610355]

=> commercial_area 테이블 완료. 이어서 인구 데이터를 적재한다.


## 7. 유동인구 / 거주인구 CSV 적재

서울시 상권분석서비스의 '길단위인구-상권'(유동인구)과 '상주인구-상권'
(거주인구) CSV를 같은 DB에 테이블로 추가한다.

두 CSV 모두 `상권_코드` 컬럼이 있어 `commercial_area` 테이블의
`trdar_cd`와 연결된다.

- CSV에는 여러 분기(2019~)가 모두 들어있다. 전체 분기를 그대로 적재한다.
  한 상권이 분기마다 한 행이므로, `(trdar_cd, quarter_code)` 조합이
  기본키가 된다.
- CSV 인코딩은 cp949다.
- DB 컬럼명은 영문으로 둔다. DB는 중간 저장소이고, 최종적으로 데이터
  소스 클래스가 fetch에서 한글 키 dict로 변환한다.

### 7-1. 경로 설정

> 길단위인구·상주인구 CSV도 raw_data 폴더에 둔다.

In [8]:
# 인구 CSV 경로 (환경에 맞게 수정)
FLOATING_CSV = "../raw_data/서울시 상권분석서비스(길단위인구-상권).csv"
RESIDENT_CSV = "../raw_data/서울시 상권분석서비스(상주인구-상권).csv"

print("유동인구 CSV:", FLOATING_CSV, "/ 존재:", os.path.exists(FLOATING_CSV))
print("거주인구 CSV:", RESIDENT_CSV, "/ 존재:", os.path.exists(RESIDENT_CSV))

유동인구 CSV: ../raw_data/서울시 상권분석서비스(길단위인구-상권).csv / 존재: True
거주인구 CSV: ../raw_data/서울시 상권분석서비스(상주인구-상권).csv / 존재: True


### 7-2. 유동인구(길단위인구) 적재

길단위인구 CSV는 27개 컬럼이다. CSV의 한글 컬럼 순서에 맞춰
영문 컬럼명을 정의하고, 그 순서대로 그대로 적재한다.

In [9]:
# 길단위인구 CSV 컬럼 순서 -> DB 영문 컬럼명
FLOATING_COLS = [
    "quarter_code",      # 기준_년분기_코드
    "trdar_se_code",     # 상권_구분_코드
    "trdar_se_name",     # 상권_구분_코드_명
    "trdar_cd",          # 상권_코드
    "trdar_name",        # 상권_코드_명
    "total",             # 총_유동인구_수
    "male",              # 남성_유동인구_수
    "female",            # 여성_유동인구_수
    "age_10",            # 연령대_10_유동인구_수
    "age_20",
    "age_30",
    "age_40",
    "age_50",
    "age_60_over",       # 연령대_60_이상_유동인구_수
    "time_00_06",        # 시간대_00_06_유동인구_수
    "time_06_11",
    "time_11_14",
    "time_14_17",
    "time_17_21",
    "time_21_24",
    "mon",               # 월요일_유동인구_수
    "tue",
    "wed",
    "thu",
    "fri",
    "sat",
    "sun",
]

# 문자열로 둘 컬럼 (나머지는 정수)
FLOATING_TEXT = {"quarter_code", "trdar_se_code", "trdar_se_name",
                 "trdar_cd", "trdar_name"}

# 테이블 생성
col_defs = []
for c in FLOATING_COLS:
    ctype = "TEXT" if c in FLOATING_TEXT else "INTEGER"
    col_defs.append(f"{c} {ctype}")

cur.execute("DROP TABLE IF EXISTS floating_population")
cur.execute(f"""
CREATE TABLE floating_population (
    {", ".join(col_defs)},
    PRIMARY KEY (trdar_cd, quarter_code)
)
""")

# CSV 적재
with open(FLOATING_CSV, encoding="cp949") as f:
    reader = csv.reader(f)
    header = next(reader)
    assert len(header) == len(FLOATING_COLS), (
        f"길단위인구 컬럼 수 불일치: CSV {len(header)} vs 정의 {len(FLOATING_COLS)}")

    placeholders = ",".join("?" * len(FLOATING_COLS))
    count = 0
    for row in reader:
        cur.execute(
            f"INSERT OR REPLACE INTO floating_population VALUES ({placeholders})",
            row)
        count += 1

conn.commit()
print("유동인구 적재 완료:", count, "행")

# trdar_cd로 빠르게 조회하도록 인덱스
cur.execute("CREATE INDEX idx_float_trdar ON floating_population(trdar_cd)")
conn.commit()
print("인덱스 생성 완료")

유동인구 적재 완료: 46184 행
인덱스 생성 완료


### 7-3. 거주인구(상주인구) 적재

상주인구 CSV는 29개 컬럼이다. 성별·연령대별 인구와 가구 수
(아파트/비아파트)를 포함한다.

In [10]:
# 상주인구 CSV 컬럼 순서 -> DB 영문 컬럼명
RESIDENT_COLS = [
    "quarter_code",      # 기준_년분기_코드
    "trdar_se_code",     # 상권_구분_코드
    "trdar_se_name",     # 상권_구분_코드_명
    "trdar_cd",          # 상권_코드
    "trdar_name",        # 상권_코드_명
    "total",             # 총_상주인구_수
    "male",              # 남성_상주인구_수
    "female",            # 여성_상주인구_수
    "age_10",            # 연령대_10_상주인구_수
    "age_20",
    "age_30",
    "age_40",
    "age_50",
    "age_60_over",
    "male_10",           # 남성연령대_10_상주인구_수
    "male_20",
    "male_30",
    "male_40",
    "male_50",
    "male_60_over",
    "female_10",         # 여성연령대_10_상주인구_수
    "female_20",
    "female_30",
    "female_40",
    "female_50",
    "female_60_over",
    "household",         # 총_가구_수
    "apartment_household",      # 아파트_가구_수
    "non_apartment_household",  # 비_아파트_가구_수
]

RESIDENT_TEXT = {"quarter_code", "trdar_se_code", "trdar_se_name",
                 "trdar_cd", "trdar_name"}

col_defs = []
for c in RESIDENT_COLS:
    ctype = "TEXT" if c in RESIDENT_TEXT else "INTEGER"
    col_defs.append(f"{c} {ctype}")

cur.execute("DROP TABLE IF EXISTS resident_population")
cur.execute(f"""
CREATE TABLE resident_population (
    {", ".join(col_defs)},
    PRIMARY KEY (trdar_cd, quarter_code)
)
""")

with open(RESIDENT_CSV, encoding="cp949") as f:
    reader = csv.reader(f)
    header = next(reader)
    assert len(header) == len(RESIDENT_COLS), (
        f"상주인구 컬럼 수 불일치: CSV {len(header)} vs 정의 {len(RESIDENT_COLS)}")

    placeholders = ",".join("?" * len(RESIDENT_COLS))
    count = 0
    for row in reader:
        cur.execute(
            f"INSERT OR REPLACE INTO resident_population VALUES ({placeholders})",
            row)
        count += 1

conn.commit()
print("거주인구 적재 완료:", count, "행")

cur.execute("CREATE INDEX idx_resid_trdar ON resident_population(trdar_cd)")
conn.commit()
print("인덱스 생성 완료")

거주인구 적재 완료: 40812 행
인덱스 생성 완료


## 8. 최종 검증 — 3개 테이블 확인

In [11]:
# 세 테이블 행 수
for table in ["commercial_area", "floating_population", "resident_population"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]}행")
print()

# 유동인구 / 거주인구의 분기 범위
cur.execute("SELECT MIN(quarter_code), MAX(quarter_code) FROM floating_population")
print("유동인구 분기 범위:", cur.fetchone())
cur.execute("SELECT MIN(quarter_code), MAX(quarter_code) FROM resident_population")
print("거주인구 분기 범위:", cur.fetchone())
print()

# 세 테이블 조인 — 상권 영역 + 최신 분기 인구
cur.execute("""
    SELECT c.trdar_name, c.trdar_se_name, f.total AS 유동인구, r.total AS 거주인구
    FROM commercial_area c
    LEFT JOIN floating_population f
        ON c.trdar_cd = f.trdar_cd AND f.quarter_code = '20254'
    LEFT JOIN resident_population r
        ON c.trdar_cd = r.trdar_cd AND r.quarter_code = '20254'
    LIMIT 5
""")
print("상권 영역 + 인구 조인 (2025년 4분기):")
for row in cur.fetchall():
    print("  ", row)

print()
print("=> 상권 단위 3개 테이블 완료. 이어서 행정동 단위 데이터를 적재한다.")

commercial_area: 1650행
floating_population: 46184행
resident_population: 40812행

유동인구 분기 범위: ('20191', '20254')
거주인구 분기 범위: ('20194', '20254')

상권 영역 + 인구 조인 (2025년 4분기):
   ('배화여자대학교(박노수미술관)', '골목상권', 1185306, 4306)
   ('종로·청계 관광특구', '관광특구', 8405499, 1767)
   ('신설종합시장', '전통시장', 43263, 2)
   ('세종마을음식문화거리(금천교시장)', '전통시장', 622776, 507)
   ('통인시장', '전통시장', 426535, 621)

=> 상권 단위 3개 테이블 완료. 이어서 행정동 단위 데이터를 적재한다.


## 9. 행정동 단위 인구 데이터 적재 (폴백용)

상권 매칭이 실패하는 건물(어느 상권에도 안 속하는 경우)을 위해, 행정동
단위 유동인구·거주인구도 적재한다.

- '길단위인구-행정동', '상주인구-행정동' CSV를 쓴다.
- 상권 버전과 컬럼이 거의 같고, 상권 식별 컬럼 자리에 행정동 코드·명이
  들어있는 점만 다르다.
- 행정동 코드는 CSV에서 8자리다. LocationInfo의 hcode(10자리)는 앞
  8자리가 이 코드와 일치하므로, 조회 시 hcode[:8]로 맞춘다.
- 테이블 이름은 상권 버전과 구분되게 `_dong` 접미사를 붙인다.

### 9-1. 행정동 CSV 경로

In [12]:
# 행정동 단위 인구 CSV 경로 (환경에 맞게 수정)
FLOATING_DONG_CSV = "../raw_data/서울시 상권분석서비스(길단위인구-행정동).csv"
RESIDENT_DONG_CSV = "../raw_data/서울시 상권분석서비스(상주인구-행정동).csv"

print("유동인구-행정동:", FLOATING_DONG_CSV, "/ 존재:", os.path.exists(FLOATING_DONG_CSV))
print("거주인구-행정동:", RESIDENT_DONG_CSV, "/ 존재:", os.path.exists(RESIDENT_DONG_CSV))

유동인구-행정동: ../raw_data/서울시 상권분석서비스(길단위인구-행정동).csv / 존재: True
거주인구-행정동: ../raw_data/서울시 상권분석서비스(상주인구-행정동).csv / 존재: True


### 9-2. 유동인구-행정동 적재

길단위인구-행정동 CSV는 25개 컬럼이다. 상권 버전(27컬럼)과 비교하면
상권 구분/코드/명 4개 대신 행정동 코드·명 2개가 들어있다.

In [13]:
# 길단위인구-행정동 CSV 컬럼 순서 -> DB 영문 컬럼명
FLOATING_DONG_COLS = [
    "quarter_code",      # 기준_년분기_코드
    "adstrd_cd",         # 행정동_코드 (8자리)
    "adstrd_name",       # 행정동_코드_명
    "total",             # 총_유동인구_수
    "male", "female",
    "age_10", "age_20", "age_30", "age_40", "age_50", "age_60_over",
    "time_00_06", "time_06_11", "time_11_14",
    "time_14_17", "time_17_21", "time_21_24",
    "mon", "tue", "wed", "thu", "fri", "sat", "sun",
]
FLOATING_DONG_TEXT = {"quarter_code", "adstrd_cd", "adstrd_name"}

col_defs = []
for c in FLOATING_DONG_COLS:
    ctype = "TEXT" if c in FLOATING_DONG_TEXT else "INTEGER"
    col_defs.append(f"{c} {ctype}")

cur.execute("DROP TABLE IF EXISTS floating_population_dong")
cur.execute(f"""
CREATE TABLE floating_population_dong (
    {", ".join(col_defs)},
    PRIMARY KEY (adstrd_cd, quarter_code)
)
""")

with open(FLOATING_DONG_CSV, encoding="cp949") as f:
    reader = csv.reader(f)
    header = next(reader)
    assert len(header) == len(FLOATING_DONG_COLS), (
        f"길단위인구-행정동 컬럼 수 불일치: "
        f"CSV {len(header)} vs 정의 {len(FLOATING_DONG_COLS)}")

    placeholders = ",".join("?" * len(FLOATING_DONG_COLS))
    count = 0
    for row in reader:
        cur.execute(
            f"INSERT OR REPLACE INTO floating_population_dong "
            f"VALUES ({placeholders})", row)
        count += 1

conn.commit()
print("유동인구-행정동 적재 완료:", count, "행")

cur.execute("CREATE INDEX idx_float_dong ON floating_population_dong(adstrd_cd)")
conn.commit()
print("인덱스 생성 완료")

유동인구-행정동 적재 완료: 11900 행
인덱스 생성 완료


### 9-3. 거주인구-행정동 적재

상주인구-행정동 CSV는 27개 컬럼이다.

In [14]:
# 상주인구-행정동 CSV 컬럼 순서 -> DB 영문 컬럼명
RESIDENT_DONG_COLS = [
    "quarter_code",      # 기준_년분기_코드
    "adstrd_cd",         # 행정동_코드 (8자리)
    "adstrd_name",       # 행정동_코드_명
    "total",             # 총_상주인구_수
    "male", "female",
    "age_10", "age_20", "age_30", "age_40", "age_50", "age_60_over",
    "male_10", "male_20", "male_30", "male_40", "male_50", "male_60_over",
    "female_10", "female_20", "female_30", "female_40", "female_50",
    "female_60_over",
    "household", "apartment_household", "non_apartment_household",
]
RESIDENT_DONG_TEXT = {"quarter_code", "adstrd_cd", "adstrd_name"}

col_defs = []
for c in RESIDENT_DONG_COLS:
    ctype = "TEXT" if c in RESIDENT_DONG_TEXT else "INTEGER"
    col_defs.append(f"{c} {ctype}")

cur.execute("DROP TABLE IF EXISTS resident_population_dong")
cur.execute(f"""
CREATE TABLE resident_population_dong (
    {", ".join(col_defs)},
    PRIMARY KEY (adstrd_cd, quarter_code)
)
""")

with open(RESIDENT_DONG_CSV, encoding="cp949") as f:
    reader = csv.reader(f)
    header = next(reader)
    assert len(header) == len(RESIDENT_DONG_COLS), (
        f"상주인구-행정동 컬럼 수 불일치: "
        f"CSV {len(header)} vs 정의 {len(RESIDENT_DONG_COLS)}")

    placeholders = ",".join("?" * len(RESIDENT_DONG_COLS))
    count = 0
    for row in reader:
        cur.execute(
            f"INSERT OR REPLACE INTO resident_population_dong "
            f"VALUES ({placeholders})", row)
        count += 1

conn.commit()
print("거주인구-행정동 적재 완료:", count, "행")

cur.execute("CREATE INDEX idx_resid_dong ON resident_population_dong(adstrd_cd)")
conn.commit()
print("인덱스 생성 완료")

거주인구-행정동 적재 완료: 11879 행
인덱스 생성 완료


## 10. 최종 검증 — 전체 5개 테이블

In [15]:
# 5개 테이블 행 수
tables = ["commercial_area",
          "floating_population", "resident_population",
          "floating_population_dong", "resident_population_dong"]
for t in tables:
    cur.execute(f"SELECT COUNT(*) FROM {t}")
    print(f"{t}: {cur.fetchone()[0]}행")
print()

# 행정동 단위 조회 테스트 — hcode 앞 8자리로
sample_hcode = "1150062000"        # 예: 공항동
dong_code = sample_hcode[:8]       # 11500620
cur.execute("""
    SELECT adstrd_name, total
    FROM floating_population_dong
    WHERE adstrd_cd = ? AND quarter_code = '20254'
""", (dong_code,))
print(f"행정동 폴백 조회 테스트 (hcode {sample_hcode} -> {dong_code}):")
print("  유동인구:", cur.fetchone())

conn.close()
print()
print("=> commercial_area.db 완성: 상권 3개 + 행정동 2개 = 5개 테이블")

commercial_area: 1650행
floating_population: 46184행
resident_population: 40812행
floating_population_dong: 11900행
resident_population_dong: 11879행

행정동 폴백 조회 테스트 (hcode 1150062000 -> 11500620):
  유동인구: ('공항동', 5445697)

=> commercial_area.db 완성: 상권 3개 + 행정동 2개 = 5개 테이블
